In [58]:
import os
import json
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely import wkt

from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage

# Find the project root (where .env_template is stored)
project_root = Path.cwd().resolve().parent

# Load environment variables
load_dotenv(project_root / ".env_template")

# Connect to Google Cloud Storage
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")

# Resolve a relative service account key path
key_path = Path(service_account_key)
if not key_path.is_absolute():
    key_path = project_root / key_path

credentials = service_account.Credentials.from_service_account_file(
    str(key_path)
)

client = storage.Client(
    project=project_id,
    credentials=credentials
)

# Find the most recently uploaded tree dataset
tree_files = [
    blob for blob in client.list_blobs(bucket_name)
    if blob.name.endswith("_tree-data.json")
]

if not tree_files:
    raise FileNotFoundError("No tree data found in bucket.")

latest_file = max(tree_files, key=lambda blob: blob.time_created)

# Retrieve tree data
tree_data = json.loads(latest_file.download_as_text())
trees_df = pd.DataFrame(tree_data)

print("Tree data retrieved successfully!")
print("File:", latest_file.name)
print("Original dataset shape:", trees_df.shape)

display(trees_df.head())

Tree data retrieved successfully!
File: 2026-10-03_11-32-48_tree-data.json
Original dataset shape: (144490, 31)


,:id,:version,:created_at,:updated_at,treeid,legalstatus,species,description,siteorder,siteinfo,...,dbhrange,cnn,roadwaydescription,onstreet,crossstreet,data_as_of,data_loaded_at,waterresponsibility,zip_code,planteddate
0,row-qb6a-ajzj~6ncv,rv-u9hi.prmg_jveq,2026-10-03T16:39:38.178Z,2026-10-03T16:39:38.178Z,226625,DPW Maintained,Leptospermum scoparium :: New Zealand Tea Tree,311 Collingwood St | Tree 2,2,Sidewalk: Property side : Yard,...,1,04272001,20TH ST to END (300 - 379),Collingwood St,20TH ST,2026-09-30,2026-10-03 09:37:43.753000-07:00,NaN,NaN,NaN
1,row-f3qm-dtp7.9ehs,rv-ydx9~fwyg~xp6r,2026-10-03T16:39:38.178Z,2026-10-03T16:39:38.178Z,21404,DPW Maintained,Pinus radiata :: Monterey Pine,2692 Sunset Blvd | Tree 6,6,Sidewalk: Curb side : Cutout,...,1,12288201,VICENTE ST to WAWONA ST (2600 - 2698) -- WEST --,SUNSET BLVD,VICENTE ST,2026-09-30,2026-10-03 09:37:43.753000-07:00,DPW,94116,NaN
2,row-vtat_eb8f_ywgv,rv-59ac~iefv.vf7h,2026-10-03T16:39:38.178Z,2026-10-03T16:39:38.178Z,113845,DPW Maintained,Ginkgo biloba :: Maidenhair Tree,2 BAY VIEW ST | Tree 1,1,Sidewalk: Curb side : Cutout,...,1,02830000,03RD ST\REVERE AVE to LATONA ST (1 - 29),BAY VIEW ST,03RD ST \ REVERE AVE,2026-09-30,2026-10-03 09:37:43.753000-07:00,Private,94124,NaN
3,row-vdn7_uj3u_3sfc,rv-s75h.q4h7~33zv,2026-10-03T16:39:38.178Z,2026-10-03T16:39:38.178Z,198376,DPW Maintained,Tristaniopsis laurina 'Elegant' :: Small-leaf ...,413 Minna St | Tree 15,15,Sidewalk: Curb side : Cutout,...,1,09042000,04TH ST to 05TH ST (300 - 399),Minna St,04TH ST,2026-09-30,2026-10-03 09:37:43.753000-07:00,NaN,94103,NaN
4,row-6bwk~q8sb~ui5s,rv-qrr2-gtp5~p44p,2026-10-03T16:39:38.178Z,2026-10-03T16:39:38.178Z,115850,DPW Maintained,Ceratonia siliqua :: Carob,3501 LAGUNA ST | Tree 5,5,Sidewalk: Curb side : Cutout,...,1,07986000,BAY ST to NORTH POINT ST (3500 - 3599),LAGUNA ST,BAY ST,2026-09-30,2026-10-03 09:37:43.753000-07:00,Private,94123,NaN


In [59]:
# Convert coordinates to numeric
trees_df['latitude'] = pd.to_numeric(
    trees_df['latitude'], errors='coerce'
)
trees_df['longitude'] = pd.to_numeric(
    trees_df['longitude'], errors='coerce'
)
# Remove trees without coordinates
trees_clean = trees_df.dropna(
    subset=['latitude', 'longitude']
).copy()
print("Original trees:", len(trees_df))
print("Trees with coordinates:", len(trees_clean))
print("Trees removed:", len(trees_df) - len(trees_clean))

Original trees: 144490
Trees with coordinates: 138738
Trees removed: 5752


In [60]:
import geopandas as gpd
from shapely import wkt

# Load police district boundaries
police_df = pd.read_csv(
    "../Current_Police_Districts_20261017.csv"
)

# Convert boundary text into geographic polygons
police_df['geometry'] = police_df['the_geom'].apply(wkt.loads)

# Create GeoDataFrame for police districts
police_gdf = gpd.GeoDataFrame(
    police_df,
    geometry='geometry',
    crs='EPSG:4326'
)

# Convert trees into geographic points
trees_gdf = gpd.GeoDataFrame(
    trees_clean,
    geometry=gpd.points_from_xy(
        trees_clean['longitude'],
        trees_clean['latitude']
    ),
    crs='EPSG:4326'
)

# Match each tree to its police district
trees_gdf = gpd.sjoin(
    trees_gdf,
    police_gdf[['district', 'geometry']],
    how='left',
    predicate='within'
)

# Rename the new district column
trees_gdf = trees_gdf.rename(
    columns={'district': 'police_district'}
)

print("Trees with a police district:", trees_gdf['police_district'].notna().sum())
print("Trees without a police district:", trees_gdf['police_district'].isna().sum())

Trees with a police district: 138735
Trees without a police district: 3


In [61]:
# Select relevant columns
tree_columns = [
    'treeid',
    'latitude',
    'longitude',
    'analysis_neighborhood',
    'supervisordistrict',
    'police_district',
    'zip_code',
    'species',
    'legalstatus',
    'planttype'
]

# Create final cleaned dataset
trees_geo_df = trees_gdf[tree_columns].copy()

print("Final cleaned dataset shape:", trees_geo_df.shape)

display(trees_geo_df.head(10))

Final cleaned dataset shape: (138738, 10)


,treeid,latitude,longitude,analysis_neighborhood,supervisordistrict,police_district,zip_code,species,legalstatus,planttype
0,226625,37.757305,-122.435712,Castro/Upper Market,8,MISSION,NaN,Leptospermum scoparium :: New Zealand Tea Tree,DPW Maintained,Tree
1,21404,37.736774,-122.494225,Sunset/Parkside,4,TARAVAL,94116,Pinus radiata :: Monterey Pine,DPW Maintained,Tree
2,113845,37.732474,-122.391891,Bayview Hunters Point,10,BAYVIEW,94124,Ginkgo biloba :: Maidenhair Tree,DPW Maintained,Tree
3,198376,37.783690,-122.404118,South of Market,6,SOUTHERN,94103,Tristaniopsis laurina 'Elegant' :: Small-leaf ...,DPW Maintained,Tree
4,115850,37.803445,-122.431796,Marina,2,NORTHERN,94123,Ceratonia siliqua :: Carob,DPW Maintained,Tree
5,250542,37.787949,-122.485235,Seacliff,1,RICHMOND,94121,Metrosideros excelsa :: New Zealand Xmas Tree,DPW Maintained,Tree
6,177372,37.730859,-122.425385,Excelsior,11,INGLESIDE,94112,Magnolia grandiflora :: Southern Magnolia,DPW Maintained,Tree
7,31001,37.709780,-122.432277,Excelsior,11,INGLESIDE,94112,Callistemon citrinus :: Lemon Bottlebrush,DPW Maintained,Tree
8,186668,37.760483,-122.496054,Sunset/Parkside,4,TARAVAL,94122,Sequoia sempervirens :: Coast Redwood,DPW Maintained,Tree
9,87175,37.733376,-122.386337,Bayview Hunters Point,10,BAYVIEW,94124,Prunus cerasifera :: Cherry Plum,DPW Maintained,Tree
